# Module 4 · Demo — LangGraph Basics

**From 0 to Agentic AI — DataHack Summit 2026**

In Module 3 we built the agent loop by hand in a raw `while`. It worked — but the guards
(step caps, routing, history) were all manual. **LangGraph** gives us those as first-class
primitives: **State**, **Nodes**, and **Edges**. This demo shows each on a tiny example
before we build the real assistant in the project notebook.

### The three primitives
- **State** — a shared object passed to every node; nodes return *updates* to it
- **Nodes** — plain Python functions: `state in → state update out`
- **Edges** — wiring between nodes; **conditional** edges route based on the state

> The whole ReAct loop from Module 3 becomes a *graph you can see and control*.

---
## Setup

In [ ]:
# Install the workshop stack (Colab). Locally, use `uv sync` instead.
# Version ranges match src/pyproject.toml (the single source of truth).
!pip install -q "langchain>=1.2,<2" "langchain-openai>=1.1,<2" \
               "langgraph>=1.0,<2" "langchain-tavily>=0.2"

In [ ]:
import os
from getpass import getpass

# Local: load keys from src/.env (walks up to find it). Colab: prompts for missing keys.
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    pass

for key in ["OPENAI_API_KEY"]:
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")

---
## Example 1 · State + nodes + a normal edge

The smallest possible graph: one piece of state, two nodes wired in a line. Notice each node
just **returns a dict that updates the state** — LangGraph merges it in.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

class State(TypedDict):
    value: int

def add_one(state: State):
    return {"value": state["value"] + 1}

def double(state: State):
    return {"value": state["value"] * 2}

g = StateGraph(State)
g.add_node("add_one", add_one)
g.add_node("double", double)
g.add_edge(START, "add_one")
g.add_edge("add_one", "double")
g.add_edge("double", END)
app = g.compile()

# Rule of thumb: whenever you compile a graph, draw it to see the flow.
display(Image(app.get_graph().draw_mermaid_png()))

print(app.invoke({"value": 10}))   # (10 + 1) * 2 = 22

---
## Example 2 · A conditional edge (branching on state)

This is where a graph beats a straight line: a **conditional edge** is a function that reads
the state and returns the *name of the next node*. That single branch is how an agent
"decides" what to do next.

In [ ]:
class NumState(TypedDict):
    n: int

def classify(state: NumState):
    return {}  # no update; this node just exists as a decision point

def route(state: NumState) -> str:
    return "big" if state["n"] >= 10 else "small"

def big(state):   print("big number:", state['n']);   return {}
def small(state): print("small number:", state['n']); return {}

g = StateGraph(NumState)
g.add_node("classify", classify)
g.add_node("big", big)
g.add_node("small", small)
g.add_edge(START, "classify")
g.add_conditional_edges("classify", route, {"big": "big", "small": "small"})
g.add_edge("big", END)
g.add_edge("small", END)
app = g.compile()

# every time we build a graph, draw it — see the branch out of `classify`
display(Image(app.get_graph().draw_mermaid_png()))

app.invoke({"n": 42})
app.invoke({"n": 3})

---
## Example 3 · `add_messages` — the reducer that grows a chat

For agents, the state is usually a **list of messages**. `add_messages` is a *reducer*: when
a node returns messages, they get **appended** to the list instead of overwriting it. This is
how the conversation (and the tool results) accumulate across the loop.

In [ ]:
from typing import Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]

def echo(state: ChatState):
    last = state["messages"][-1].content
    return {"messages": [AIMessage(content=f"you said: {last}")]}

g = StateGraph(ChatState)
g.add_node("echo", echo)
g.add_edge(START, "echo")
g.add_edge("echo", END)
app = g.compile()

display(Image(app.get_graph().draw_mermaid_png()))

out = app.invoke({"messages": [HumanMessage(content="hello")]})
for m in out["messages"]:
    m.pretty_print()

> Notice the output has **both** messages — the human one we sent *and* the AI reply. The reducer appended; it didn't replace. That accumulation is the agent's short-term memory.

---
## Example 4 · Loops &amp; termination — what stops an infinite loop?

A graph edge can point **backwards**, which is how agents loop. But a loop needs a way to *stop*.
In Module 3 we used a manual `max_steps` cap. LangGraph gives you the same guard for free:
**`recursion_limit`** — hit it, and the run raises `GraphRecursionError` instead of running forever.

In [ ]:
class LoopState(TypedDict):
    count: int

def tick(state: LoopState):
    return {"count": state["count"] + 1}

def keep_going(state: LoopState) -> str:
    # stop after 3 ticks; otherwise loop back
    return "end" if state["count"] >= 3 else "loop"

g = StateGraph(LoopState)
g.add_node("tick", tick)
g.add_edge(START, "tick")
g.add_conditional_edges("tick", keep_going, {"loop": "tick", "end": END})
app = g.compile()

display(Image(app.get_graph().draw_mermaid_png()))

# A proper termination condition -> stops cleanly at 3:
print("clean stop:", app.invoke({"count": 0}))

Now the safety net. If the stop condition were buggy and the graph looped forever, `recursion_limit` cuts it off — pass it at invoke time:

In [ ]:
from langgraph.errors import GraphRecursionError

# Force a runaway loop (always 'loop') to show the guard firing:
def never_stop(state: LoopState) -> str:
    return "loop"

g2 = StateGraph(LoopState)
g2.add_node("tick", tick)
g2.add_edge(START, "tick")
g2.add_conditional_edges("tick", never_stop, {"loop": "tick", "end": END})
runaway = g2.compile()

try:
    runaway.invoke({"count": 0}, {"recursion_limit": 5})
except GraphRecursionError as e:
    print("stopped by recursion_limit:", e)

> **Two layers of protection.** Your own **stop condition** (a conditional edge to `END`) is how the agent *should* finish; **`recursion_limit`** is the backstop for when it doesn't. Always have both.

---
## Example 5 · Retries — routing to a fallback node

Real steps fail — a flaky API, a bad tool result. Because routing is just a conditional edge, a
**retry** is nothing new: on failure, route **back** to the node (or to a *fallback* node) instead
of crashing the run.

In [ ]:
class JobState(TypedDict):
    attempts: int
    ok: bool

def do_work(state: JobState):
    # pretend the work fails the first 2 times, then succeeds
    attempts = state["attempts"] + 1
    return {"attempts": attempts, "ok": attempts >= 3}

def check(state: JobState) -> str:
    if state["ok"]:
        return "done"
    return "retry" if state["attempts"] < 5 else "give_up"

def give_up(state): print("giving up after", state['attempts'], "tries"); return {}

g = StateGraph(JobState)
g.add_node("do_work", do_work)
g.add_node("give_up", give_up)
g.add_edge(START, "do_work")
g.add_conditional_edges("do_work", check,
                        {"retry": "do_work", "give_up": "give_up", "done": END})
g.add_edge("give_up", END)
app = g.compile()

display(Image(app.get_graph().draw_mermaid_png()))
print(app.invoke({"attempts": 0, "ok": False}))   # succeeds on attempt 3

> The retry loop (`do_work → check → do_work`) is capped by the same `attempts < 5` idea, with a `give_up` fallback. Same three primitives — no new machinery.

---
## Key takeaways
- **State** flows through the graph; nodes return **partial updates**, merged by reducers.
- **Conditional edges** route on the state — this is where branching, loops, and retries all live.
- **`add_messages`** appends to a message list — the backbone of an agent's memory.
- **Termination:** a stop-condition edge to `END`, plus **`recursion_limit`** as a backstop.
- **Retries:** on failure, route back to a node or to a **fallback** — just another conditional edge.

➡️ **Next (the project notebook):** assemble these into a real **agent loop** — model node,
tool node, and a conditional edge that loops back — the **Knowledge Assistant v1**.